In [ ]:
# Station request parameters that Client.get_stations() can accept
params = dict(
    channel="BHZ", 
    starttime="1995-11-14T06:32:55.750000Z", 
    endtime="1995-11-14T06:35:55.750000Z", 
    level="channel",
    includeoverlaps="false",
    nodata=404,
)

In [ ]:
from obspy.clients.fdsn import RoutingClient
client = RoutingClient(
    "iris-federator", 
    debug=True, 
    timeout=10,
    include_providers=None,
    exclude_providers=None,
    credentials=None,
)

# download the list of channels
r = client._download(url=client._url + "/query", params=params)
# split the responses for multiple datacenters
records = client._split_routing_response(r.text, service="station")
# remove data centers that we don't need
records = client._filter_requests(records)

In [ ]:
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoServiceException, FDSNNoDataException, FDSNTimeoutException
from pathlib import Path


Path("stations").mkdir(exist_ok=True, parents=True)
Path("mseed").mkdir(exist_ok=True, parents=True)


from multiprocessing.dummy import Pool as ThreadPool

def _try_download_bulk(datacenter, bulk):   
    try:
        client = Client(datacenter, timeout=120)
    except FDSNNoServiceException:
        print("No service for", datacenter)
        return None, None
    try:
        inv = client.get_stations_bulk(bulk=records[datacenter], level="response")
        st = client.get_waveforms_bulk(bulk=records[datacenter])
        print(len(st))
        return inv, st
    except (FDSNNoDataException, FDSNTimeoutException, FDSNBadGatewayException, ValueError):
        return None, None
    
pool = ThreadPool(processes=len(records.keys()))
args = [(datacenter, records[datacenter]) for datacenter in records.keys()]
pool.starmap(_try_download_bulk, args)      
pool.close() 